In [1]:
import os
import time
import logging
import zipfile
import shutil
import requests
import pandas as pd
from io import StringIO
from datetime import datetime

In [2]:
# ==========================================
# CONFIGURACION GLOBAL
# ==========================================

DATA_DIR = "datasets_kaggle"
LOG_DIR = "logs_kaggle"
os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(LOG_DIR, exist_ok=True)

# Configuracion del archivo de log
LOG_FILE = os.path.join(LOG_DIR, "ingesta.log")
logging.basicConfig(
    filename=LOG_FILE,
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S"
)

# Ruta al archivo kaggle.json
KAGGLE_JSON_PATH = os.path.normpath(os.path.join(os.getcwd(), '..', 'autentificacion', 'kaggle.json'))
if not os.path.exists(KAGGLE_JSON_PATH):
    raise FileNotFoundError(f"No se encontro kaggle.json en {KAGGLE_JSON_PATH}")

# Variables de entorno para la API de Kaggle
os.environ["KAGGLE_CONFIG_DIR"] = os.path.dirname(KAGGLE_JSON_PATH)
os.environ["KAGGLE_CONFIG_PATH"] = KAGGLE_JSON_PATH

In [3]:
# ==========================================
# FUNCIONES AUXILIARES
# ==========================================

def guardar_csv(df, nombre_archivo):
    """Guarda un DataFrame como archivo CSV, reemplazando si ya existe."""
    output_path = os.path.join(DATA_DIR, nombre_archivo)
    if os.path.exists(output_path):
        os.remove(output_path)
        logging.warning(f"Archivo anterior eliminado: {nombre_archivo}")
    df.to_csv(output_path, index=False)
    logging.info(f"Archivo guardado correctamente: {output_path}")
    logging.info(f"{nombre_archivo}: {df.shape[0]} filas y {df.shape[1]} columnas.")


In [4]:
# ==========================================
# DESCARGA DESDE KAGGLE
# ==========================================

def descargar_kaggle(dataset, nombre_archivo):
    """Descarga un dataset desde Kaggle y lo guarda como CSV."""
    from kaggle.api.kaggle_api_extended import KaggleApi
    print(f"Iniciando descarga de {dataset} desde Kaggle...")
    try:
        api = KaggleApi()
        api.authenticate()
        inicio = time.time()
        api.dataset_download_files(dataset, path=DATA_DIR, unzip=False)
        logging.info(f"Descarga iniciada desde Kaggle: {dataset}")

        zip_file = next((f for f in os.listdir(DATA_DIR) if f.endswith('.zip')), None)
        if not zip_file:
            raise FileNotFoundError("No se encontro el archivo ZIP descargado de Kaggle.")

        zip_path = os.path.join(DATA_DIR, zip_file)
        with zipfile.ZipFile(zip_path, "r") as z:
            z.extractall(DATA_DIR)
        os.remove(zip_path)

        csv_files = [f for f in os.listdir(DATA_DIR) if f.endswith('.csv')]
        if not csv_files:
            raise FileNotFoundError("No se encontro ningun CSV dentro del dataset Kaggle.")

        csv_path = os.path.join(DATA_DIR, csv_files[0])
        df = pd.read_csv(csv_path)
        guardar_csv(df, nombre_archivo)
        os.remove(csv_path)

        fin = time.time()
        logging.info(f"Descarga completada: {dataset} en {fin - inicio:.2f} segundos")
        print(f"Descarga finalizada desde Kaggle: {dataset}")

    except Exception as e:
        logging.error(f"Error en la descarga de Kaggle: {e}")
        print(f"Error en descarga Kaggle: {e}")


In [5]:
# ==========================================
# DESCARGA DESDE KAGGLEHUB
# ==========================================

def descargar_kagglehub():
    """Descarga un dataset usando kagglehub y lo guarda como Personas.csv."""
    import kagglehub
    dataset_handle = "missionjee/car-sales-report"
    print(f"Iniciando descarga desde KaggleHub: {dataset_handle}...")
    try:
        inicio = time.time()
        local_path = kagglehub.dataset_download(dataset_handle)
        logging.info(f"Dataset descargado con kagglehub: {dataset_handle}")

        csv_file = None
        for root, dirs, files in os.walk(local_path):
            for f in files:
                if f.lower().endswith(".csv"):
                    csv_file = os.path.join(root, f)
                    break
            if csv_file:
                break

        if csv_file is None:
            raise FileNotFoundError("No se encontro ningun archivo CSV en el dataset descargado por kagglehub.")

        final_csv_path = os.path.join(DATA_DIR, "Personas.csv")
        shutil.copy(csv_file, final_csv_path)
        logging.info(f"CSV guardado en: {final_csv_path}")

        kagglehub_temp_folder = os.path.join(os.path.expanduser("~"), ".cache", "kagglehub", "datasets")
        if os.path.exists(kagglehub_temp_folder):
            shutil.rmtree(kagglehub_temp_folder)
            logging.warning(f"Carpeta temporal eliminada: {kagglehub_temp_folder}")

        fin = time.time()
        logging.info(f"Descarga con kagglehub completada en {fin - inicio:.2f} segundos")
        print(f"Descarga finalizada desde KaggleHub: {dataset_handle}")

    except Exception as e:
        logging.error(f"Error durante la descarga con kagglehub: {e}")
        print(f"Error en descarga KaggleHub: {e}")


In [6]:
# ==========================================
# DESCARGA DESDE GITHUB
# ==========================================

def descargar_github():
    """Descarga un archivo CSV desde un repositorio de GitHub."""
    url = "https://raw.githubusercontent.com/Fuffi0901/MarcasCoches/refs/heads/main/InfoMarcas.csv"
    print("Iniciando descarga desde GitHub...")
    try:
        inicio = time.time()
        response = requests.get(url)
        with open(os.path.join(DATA_DIR, "InfoMarcas.csv"), "wb") as f:
            f.write(response.content)
        fin = time.time()
        logging.info(f"Descarga desde GitHub completada en {fin - inicio:.2f} segundos")
        print("Descarga finalizada desde GitHub")
    except Exception as e:
        logging.error(f"Error en la descarga desde GitHub: {e}")
        print(f"Error en descarga GitHub: {e}")

In [7]:
# ==========================================
# DESCARGA DESDE OPENDATASOFT
# ==========================================

def descargar_opendatasoft():
    """Descarga un dataset desde OpenDataSoft y lo guarda como ModelosCoches.csv."""
    domain = "public.opendatasoft.com"
    dataset_id = "all-vehicles-model"
    url = f"https://{domain}/api/explore/v2.1/catalog/datasets/{dataset_id}/exports/csv"
    params = {
        "select": "*",
        "limit": "-1",
        "timezone": "UTC",
        "use_labels": "true",
        "delimiter": ";"
    }
    print("Iniciando descarga desde OpenDataSoft...")
    try:
        inicio = time.time()
        response = requests.get(url, params=params)
        response.raise_for_status()
        df = pd.read_csv(StringIO(response.text), sep=";")
        guardar_csv(df, "ModelosCoches.csv")
        fin = time.time()
        logging.info(f"Descarga desde OpenDataSoft completada en {fin - inicio:.2f} segundos")
        print("Descarga finalizada desde OpenDataSoft")
    except Exception as e:
        logging.error(f"Error en descarga OpenDataSoft: {e}")
        print(f"Error en descarga OpenDataSoft: {e}")


In [8]:
# ==========================================
# EJECUCION PRINCIPAL
# ==========================================

if __name__ == "__main__":
    print("Inicio del proceso de ingesta unificada...")
    logging.info("Proceso de ingesta unificada iniciado")
    descargar_kaggle("syedanwarafridi/vehicle-sales-data", "CochesVentas.csv")
    descargar_kagglehub()
    descargar_github()
    descargar_opendatasoft()
    logging.info("Ingesta completada correctamente")
    print("Proceso de ingesta completado")

Inicio del proceso de ingesta unificada...
Iniciando descarga de syedanwarafridi/vehicle-sales-data desde Kaggle...
Dataset URL: https://www.kaggle.com/datasets/syedanwarafridi/vehicle-sales-data
Descarga finalizada desde Kaggle: syedanwarafridi/vehicle-sales-data


c:\Users\iabd\AppData\Local\anaconda3\envs\Prueba1\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Iniciando descarga desde KaggleHub: missionjee/car-sales-report...


100%|██████████| 657k/657k [00:00<00:00, 1.32MB/s]

Extracting files...


Descarga finalizada desde KaggleHub: missionjee/car-sales-report
Iniciando descarga desde GitHub...
Descarga finalizada desde GitHub
Iniciando descarga desde OpenDataSoft...
Descarga finalizada desde OpenDataSoft
Proceso de ingesta completado
